# 01. Data and EDA

Milestone 1: the data the forecasts will be built on, and what it looks like.

1. **Load.** Real data if `$VOL_DATA_DIR` holds the pipeline's output and
   `configs/protocol.yaml` has a frozen `data_end`; otherwise seeded synthetic
   data with known parameters. The next cell prints which.
2. **Check** that every session of the master calendar 𝒯 has a VIX close and an
   S&P 500 close (the gap policy, ADR 0001 D4.1 amended M1).
3. **Explore**: (a) volatility clustering, (b) persistence, (c) the AR(1)
   half-life, (d) unit-root tests, (e) the implied-minus-realised spread,
   (f) vol-of-vol, (g) the term structure.

**No holdout row is looked at.** Every statistic here uses sessions before
`holdout_start` only (`ivf.data.process.eda_inputs`): the holdout is scored once,
with hyper-parameters frozen (ADR 0001 4.6, amended M1, review A). The samples
are `pre_holdout` (burn-in and dev together), `burn_in` and `dev`.

**Licensing.** Cboe and Yahoo data are for personal, non-commercial use. This
notebook is committed with its outputs stripped (`tools/build_notebooks.py
--check` fails on a notebook saved with outputs). Real-data numbers you see
here stay on your machine (ADR 0001 D2): do not paste them into the repo.

To fill `$VOL_DATA_DIR` with real data:

```
uv run python -m ivf.data fetch
uv run python -m ivf.data build-interim
uv run python -m ivf.data verify
uv run python -m ivf.data check --freeze-data-end
```

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

In [ ]:
import os
import warnings

import numpy as np

import pandas as pd

from ivf import config, eda, protocol
from ivf.data import build_interim, calendar, check, process, synthetic
from ivf.features import build

warnings.filterwarnings("ignore", category=FutureWarning)


def load():
    """(mode, protocol, iv, price, build_report): real if available, else synthetic."""
    if os.environ.get("IVF_NOTEBOOK_DATA") != "synthetic":
        try:
            proto = protocol.load_protocol(protocol.protocol_path())
            proto.require_data_end()
            iv, price, report = build_interim.read_interim(config.data_layout().root)
            return "real", proto, iv, price, report
        except (protocol.ProtocolError, build_interim.InterimError) as why:
            print("no real data:", why)
    proto = protocol.load_protocol(protocol.protocol_path(synthetic=True))
    iv, price = synthetic.synthetic_market(proto)
    report = {"files": [{"symbol": s, "fetched_at": "2026-10-27T12:00:00.000000Z", "duplicates_dropped": 0, "duplicate_dates": []}
                        for s in list(iv["symbol"].unique()) + ["^GSPC"]]}
    return "synthetic", proto, iv, price, report


MODE, PROTO, IV, PRICE, REPORT = load()
if MODE == "real":
    print("data: REAL (Cboe, Yahoo). Local only: never commit these numbers (ADR 0001 D2).")
else:
    params = synthetic.SyntheticParams()
    print(f"data: SYNTHETIC (seed {params.seed}). Known truth: phi = {params.phi}, "
          f"half-life = {params.half_life:.2f} sessions, GARCH alpha = {params.alpha}, beta = {params.beta}.")
print("protocol:", PROTO.path.name, "| train_start", PROTO.train_start, "| holdout_start", PROTO.holdout_start,
      "| data_end", PROTO.data_end)

## 1. The master calendar and the gap check

𝒯 is the NYSE session calendar, computed from dates alone. `data_check` looks
for a VIX close and an S&P 500 close on every session from `train_start` to
`data_end`. A missing one is a *core gap*: it stays null, is never filled, and
is allowed only in burn-in, far enough before the first fold cutoff that no
forecast origin can see it (gap policy G1 to G5 in `ivf/data/check.py`). The
frozen `data_end` must also match the freeze record that
`check --freeze-data-end` wrote. If anything is VIOLATED, the cell stops the
notebook.

In [ ]:
RESULT = check.frozen_check(PROTO, IV, PRICE, REPORT)  # the data and the freeze record (amended M1, review B)
check.print_summary(RESULT, PROTO, PROTO.data_end)
if not RESULT.ok:
    # Review B: the first version printed VIOLATED and went on to compute every statistic below.
    raise RuntimeError("check reports VIOLATED (above): nothing below is computed on data the protocol does not support")

CAL = calendar.master_calendar(PROTO)
# The EDA sees sessions and rows before holdout_start only: no origin or target in the holdout.
DATES, PERIODS, EDA_IV, EDA_PRICE = process.eda_inputs(CAL, IV, PRICE, PROTO.holdout_start)
FEATURES = build.build_features(DATES, EDA_IV, EDA_PRICE)
print(len(DATES), "sessions before holdout_start", PROTO.holdout_start, "(the holdout is not looked at in M1)")
print(pd.Series(PERIODS).value_counts().to_dict())
SAMPLES = {
    "pre_holdout": np.ones(len(DATES), dtype=bool),
    "burn_in": np.array(PERIODS) == "burn_in",
    "dev": np.array(PERIODS) == "dev",
}

## 2. (a) Volatility clustering

Daily S&P 500 log returns `r` are close to uncorrelated, but `r**2` and `|r|`
are strongly autocorrelated: big moves follow big moves. Ljung-Box tests "no
autocorrelation up to lag m"; a tiny p-value rejects it. See
`docs/concepts/04-volatility-clustering-and-the-acf.md`.

In [ ]:
r = FEATURES["r_spx"].to_numpy()
for name, series in (("r", r), ("r^2", r**2), ("|r|", np.abs(r))):
    lb = eda.acf_ljung_box(series, 21)
    print(f"{name:4} acf lag 1,5,21 = {lb.acf[0]:+.3f} {lb.acf[4]:+.3f} {lb.acf[20]:+.3f}   "
          f"Ljung-Box Q(21) = {lb.q[-1]:9.1f}  p = {lb.p_value[-1]:.2g}")

## 3. (b, c) Persistence and the AR(1) half-life

VIX is very persistent: its ACF decays slowly. An AR(1) on ln VIX,
`x_t = c + phi*x_{t-1} + e_t`, says a deviation from the mean shrinks by `phi`
each session, so it halves after `ln(0.5)/ln(phi)` sessions. The interval maps
phi's 95% interval through that formula. See
`docs/concepts/05-persistence-half-life-and-unit-roots.md`.

In [ ]:
log_vix = FEATURES["log_vix"].to_numpy()
vix = np.exp(log_vix)
print("ACF of VIX    at lags 1, 5, 21:", np.round(eda.acf(vix, 21)[[1, 5, 21]], 3))
print("ACF of ln VIX at lags 1, 5, 21:", np.round(eda.acf(log_vix, 21)[[1, 5, 21]], 3))
for name, mask in SAMPLES.items():
    hl = eda.ar1_half_life(np.where(mask, log_vix, np.nan))
    print(f"{name:11} phi = {hl.phi:.4f} (95% {hl.phi_ci[0]:.4f}..{hl.phi_ci[1]:.4f})  "
          f"half-life = {hl.half_life:.1f} sessions (95% {hl.half_life_ci[0]:.1f}..{hl.half_life_ci[1]:.1f}), n = {hl.n}")

## 4. (d) Mean reversion: ADF and KPSS

The two tests have **opposite nulls**. ADF's null is a unit root (no mean
reversion); rejecting it supports mean reversion. KPSS's null is stationarity;
rejecting it argues against. "Stationary at 5%" means ADF rejects and KPSS
does not. For a series as persistent as ln VIX, KPSS rejects a true stationary
AR(1) far more often than 5% (measured in the concept note), so read a
disagreement with care. Both tests run on the longest stretch of sessions with
no gap (gap policy G2: no value is paired across a gap).

In [ ]:
ur = eda.unit_root_tests(log_vix)
print(f"tested {ur.n} consecutive sessions, {DATES[ur.segment[0]]}..{DATES[ur.segment[1] - 1]}, of {ur.n_nonmissing} with a value")
print(eda.UnitRoot.ADF_NULL)
print(f"  ADF stat {ur.adf_stat:.2f}, p = {ur.adf_p:.3g}, lags {ur.adf_lags}")
print(eda.UnitRoot.KPSS_NULL)
print(f"  KPSS stat {ur.kpss_stat:.3f}, p {ur.kpss_p_bound or '='} {ur.kpss_p:.3g}")
print("stationary at 5% (ADF rejects and KPSS does not):", ur.stationary_at_5pct)

## 5. (e) Implied minus realised: the spread

VIX at t against the realised volatility of the **next** 21 sessions (the
`rv21` target, ADR 0001 4.2). In variance units the average gap is the
ex-post variance risk premium. This looks forward, which is fine for
describing the past and forbidden as a model input (rule L1). On synthetic
data VIX and returns are independent, so the gap is just a level difference.
See `docs/concepts/06-variance-risk-premium-and-vol-of-vol.md`.

In [ ]:
rv21_fwd = build.forward(FEATURES["rv21_back"].to_numpy(), build.RV_WINDOW)  # null where t+21 is in the holdout
for name, mask in SAMPLES.items():
    s = eda.iv_rv_spread(vix, rv21_fwd, mask)
    print(f"{name:11} n = {s.n:5}  mean gap {s.mean_vol_points:+.2f} vol pts (median {s.median_vol_points:+.2f})  "
          f"variance gap {s.mean_variance:+.5f}  implied above realised on {s.share_positive:.1%} of origins")

## 6. (f) Vol-of-vol

The annualised rolling sd of daily changes in ln VIX, and whether larger
levels come with larger *next* shocks: the rank correlation of the level with
the size of the next AR(1) shock to ln VIX. The raw next move would mislead,
because it contains the pull back to the mean, which is larger far from it.
Its p-value is a bootstrap that refits the AR(1) on every resampled path (the
usual Spearman p-value treats the fitted shocks as known and rejects far too
often when shocks are skewed) and resamples the shocks in blocks of
consecutive sessions, so that volatility clustering is kept: drawn one at a
time, the shocks lose it, and the test rejected on 47% of GARCH paths with no
level effect. VVIX, Cboe's implied vol of VIX, is compared where it exists.

In [ ]:
vvix = build.align(DATES, EDA_IV, "symbol", "VVIX") if (EDA_IV["symbol"] == "VVIX").any() else None
vov = eda.vol_of_vol(vix, 21, vvix)
print(f"mean annualised vol of ln VIX (21-session windows): {vov.mean_vov:.3f}")
print(f"Spearman(V_t, |next AR(1) shock|) = {vov.spearman_level_vs_next_abs_resid:+.3f} "
      f"(block-bootstrap p {vov.e6_p_bootstrap:.2g}, {vov.e6_n_boot} paths, blocks of {vov.e6_block})   <- the E6 measure")
print(f"Spearman(V_t, |d ln V_t+1|)       = {vov.spearman_level_vs_next_abs_dlog:+.3f}  (includes the mean-reversion pull)")
print(f"Spearman(V_t, |d V_t+1|)          = {vov.spearman_level_vs_next_abs_dpoints:+.3f}  (in points: rises with the level by construction)")
print(f"Spearman(VIX, VVIX)         = {vov.spearman_vix_vs_vvix:+.3f}")

## 7. (g) The term structure

Contango means VIX at or below VIX3M (longer-dated implied vol higher); the
regime rule calls it backwardation only when VIX > VIX3M. The slope
`ln(VIX3M/VIX)` over time shows the inversions in stress.

In [ ]:
import matplotlib.pyplot as plt

levels = {s: build.align(DATES, EDA_IV, "symbol", s) for s in ("VIX9D", "VIX", "VIX3M", "VIX6M", "VIX1Y")}
for name, mask in SAMPLES.items():
    ts = eda.term_structure_summary(levels, mask=mask)
    print(f"{name:11} n = {ts.n:5}  contango share {ts.contango_share:.1%}  mean ln(VIX3M/VIX) {ts.mean_log_slope:+.3f}  "
          f"whole curve upward on {ts.upward_full_curve_share:.1%} of {ts.n_full_curve} sessions")

slope = np.log(levels["VIX3M"] / levels["VIX"])
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(DATES, slope, lw=0.6)
ax.axhline(0, color="k", lw=0.8)
ax.set_title(f"ln(VIX3M / VIX), {MODE} data: below 0 is backwardation")
fig.tight_layout()  # Jupyter draws the figure at the end of the cell; no plt.show(), which blocks outside it